# Generative Sequential Recommendation — Full Pipeline (Colab)

End-to-end run on Amazon Beauty: data → embeddings → RQ-VAE → Semantic IDs → T5 training → evaluation, plus the SASRec baseline.

**Setup**: Runtime → Change runtime type → **GPU** (A100 recommended; T4 works if you halve `batch_size`).

**Inputs**
- `data/beauty_data.pkl` is in the repo; clone is enough.
- `embedding/item_embeddings_raw.npy` (~74 MB) must be uploaded — it's gitignored.  
  Generate it locally first with `python embedding/extract_embeddings.py` (needs Ollama)
  if you don't already have it.

**Outputs**
- `checkpoints/rqvae_best.pt`
- `embedding/semantic_ids_rqvae.npy`
- `checkpoints/best_model_t5.pt`
- `checkpoints/sasrec_best.pt`

Each stage is independent; skip the cells you don't need.

## 1. Environment

In [ ]:
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {name}  ({mem:.1f} GB)')
else:
    print('No GPU detected. Runtime → Change runtime type → GPU.')

In [ ]:
!pip install transformers scikit-learn -q
print('deps installed')

In [ ]:
import os
REPO = 'Generative-Sequential-Recommendation'
if not os.path.exists(REPO):
    !git clone https://github.com/rhine-yangrui/Generative-Sequential-Recommendation.git
else:
    !cd {REPO} && git pull
%cd {REPO}
!git log --oneline -3

## 2. Upload item embeddings (~74 MB)

Skip this cell if you only want to run SASRec.

In [ ]:
import os
from google.colab import files

os.makedirs('embedding', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

print('Upload item_embeddings_raw.npy from your local embedding/ folder')
uploaded = files.upload()
for fname in uploaded:
    if fname != 'item_embeddings_raw.npy':
        print(f'unexpected filename: {fname!r}')
        continue
    dest = f'embedding/{fname}'
    os.rename(fname, dest)
    print(f'moved -> {dest}  ({os.path.getsize(dest)/1e6:.1f} MB)')

import numpy as np
raw = np.load('embedding/item_embeddings_raw.npy', allow_pickle=True).item()
vals = list(raw.values())
print(f'loaded: {len(raw)} items x {vals[0].shape[0]}d')

## 3. RQ-VAE → Semantic IDs (~1–2 h on A100)

Hyperparameters live in `embedding/rqvae.py` (`K_LEVELS=[256,256,256]`, batch=1024, lr=1e-3, 3000 epochs).

In [ ]:
!python embedding/rqvae.py

In [ ]:
!python embedding/generate_rqvae_ids.py

import numpy as np
sids = np.load('embedding/semantic_ids_rqvae.npy', allow_pickle=True).item()
vals = list(sids.values())
unique = len({tuple(v) for v in vals})
print(f'{len(sids)} items, unique 4-tuples: {unique}')

### Optional safety net: download the RQ-VAE artifacts

If T5 training (next stage) hits a long Colab disconnect, you can resume by re-uploading these two files instead of re-running RQ-VAE.

In [ ]:
from google.colab import files
import os
for fpath in ['checkpoints/rqvae_best.pt', 'embedding/semantic_ids_rqvae.npy']:
    if os.path.exists(fpath):
        files.download(fpath)
        print(f'{fpath}  ({os.path.getsize(fpath)/1e6:.1f} MB)')

## 4. Train and evaluate the T5 generative recommender (~2–3 h on A100)

Hyperparameters live in `train.py` (`CONFIG`). Lower `batch_size` from 512 to 256 if VRAM is tight.

In [ ]:
!python train.py

In [ ]:
!python evaluate.py

## 5. SASRec baseline

Independent of the generative path; needs only `beauty_data.pkl`.

In [ ]:
!python baseline/sasrec_train.py

## 6. Download checkpoints (optional)

In [ ]:
import os
from google.colab import files

for fpath in [
    'checkpoints/best_model_t5.pt',
    'checkpoints/sasrec_best.pt',
]:
    if os.path.exists(fpath):
        files.download(fpath)
        print(f'{fpath}  ({os.path.getsize(fpath)/1e6:.1f} MB)')